# DoSA-Maxwell 통합 자동화 튜토리얼

이 노트북은 DoSA 설계 파일을 파싱하고, Ansys Maxwell 모델링 커맨드를 자동 생성하는 전체 파이프라인을 시연합니다.

## 워크플로우
1. 패키지 로드 및 샘플 설계 파싱
2. 형상/재질/여자 추출 확인
3. Maxwell 빌드 (커맨드 로그 모드)
4. 프로파일 선택 및 PDF 기반 프리셋 비교
5. Unified Plan 상태 확인

In [2]:
from pathlib import Path
import json, sys

ROOT = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/31_DoSA-Maxwell-Automation")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import dosa_maxwell
print(f"dosa_maxwell loaded from: {dosa_maxwell.__file__}")
print(f"Available API: {dosa_maxwell.__all__}")

dosa_maxwell loaded from: E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\src\dosa_maxwell\__init__.py
Available API: ['DesignModel', 'Geometry2D', 'MaxwellSessionBuilder', 'NodeModel', 'TestModel', 'apply_dosa_to_maxwell', 'extract_geometry', 'geometry_from_coil_params', 'get_profile', 'get_unified_plan_summary', 'list_profiles', 'parse_dosa_file', 'resolve_magnet_direction', 'resolve_material']


## 1. 샘플 설계 파싱 (Solenoid / VCM)

In [ ]:
from dosa_maxwell import parse_dosa_file, extract_geometry, resolve_material

SOLENOID = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples/Solenoid/Solenoid.dsa")
VCM = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples/VCM/VCM.dsa")

design_sol = parse_dosa_file(SOLENOID)
design_vcm = parse_dosa_file(VCM)

print("=== Solenoid ===")
print(f"  Parts: {[p.name for p in design_sol.parts]}")
print(f"  Tests: {[t.name for t in design_sol.tests]}")

print("\n=== VCM ===")
print(f"  Parts: {[p.name for p in design_vcm.parts]}")
print(f"  Tests: {[t.name for t in design_vcm.tests]}")

## 2. 형상 및 재질 추출

In [ ]:
print("--- Solenoid parts geometry & material ---")
for part in design_sol.parts:
    geom = extract_geometry(part)
    mat = resolve_material(part.properties.get('Material', 'Air'))
    valid = geom.is_valid if geom else False
    pts = len(geom.points) if geom else 0
    print(f"  {part.name:10s} | maxwell_mat={mat.maxwell_name:25s} | vertices={pts} | valid={valid}")

In [ ]:
from dosa_maxwell import resolve_magnet_direction

print("--- VCM parts (magnet direction 포함) ---")
for part in design_vcm.parts:
    geom = extract_geometry(part)
    mat = resolve_material(part.properties.get('Material', 'Air'))
    info = f"  {part.name:10s} | {mat.maxwell_name:15s} | {mat.category}"
    if part.kind == 'Magnet':
        direction = part.properties.get('MagnetDirection', '')
        vec = resolve_magnet_direction(direction)
        info += f" | dir={direction} -> vec={vec}"
    print(info)

## 3. Maxwell 빌드 — WS01 프리셋으로 Solenoid

In [ ]:
from dosa_maxwell import MaxwellSessionBuilder, get_profile

profile_ws01 = get_profile("ws01_2020r1")
print(f"Profile: {profile_ws01.name}")
print(f"  Solution: {profile_ws01.solution_type}")
print(f"  Time step: {profile_ws01.time_step}, Stop: {profile_ws01.stop_time}")
print(f"  Source PDF: {profile_ws01.source_pdf}")

out_sol = ROOT / "output" / "tutorial_solenoid"
builder = MaxwellSessionBuilder(
    design=design_sol,
    profile=profile_ws01,
    out_dir=out_sol,
    mode="2d",
)
# live=True: 실제 AEDT 세션에 프로젝트 생성 (AEDT 실행 필요)
result = builder.build(live=True)

print(f"\nBuild result: ok={result.ok}, commands={len(result.commands)}, errors={len(result.errors)}")
if result.errors:
    print("\nErrors:")
    for e in result.errors:
        print(f"  - {e}")

In [ ]:
# 생성된 커맨드 로그 확인
cmd_log = json.loads((out_sol / "maxwell_commands.json").read_text(encoding="utf-8"))

print(f"Total commands: {len(cmd_log['commands'])}")
print(f"Profile used: {cmd_log['profile']['name']} ({cmd_log['profile']['source_pdf']})")
print("\nCommand sequence:")
for i, cmd in enumerate(cmd_log['commands'], 1):
    args_summary = ', '.join(f"{k}={v}" for k, v in cmd['args'].items() if k != 'points')
    print(f"  {i:2d}. {cmd['method']:30s} | {args_summary}")

## 4. PDF 프리셋 비교 — LE01로 VCM 빌드

In [ ]:
from dosa_maxwell import list_profiles

print("Available profiles (from unified_plan.json):")
for p in list_profiles():
    print(f"  {p['name']:15s} | {p['solution_type']:15s} | step={p['time_step']} | stop={p['stop_time']} | pdf={p['source_pdf']}")

# LE01 프리셋으로 VCM 빌드
profile_le01 = get_profile("le01_2020r1")
out_vcm = ROOT / "output" / "tutorial_vcm"
builder_vcm = MaxwellSessionBuilder(
    design=design_vcm, profile=profile_le01, out_dir=out_vcm, mode="2d"
)
result_vcm = builder_vcm.build(live=False)

print(f"\nVCM build (LE01): ok={result_vcm.ok}, commands={len(result_vcm.commands)}")
print("\nAssignment commands:")
for cmd in result_vcm.commands:
    if cmd.method.startswith('assign'):
        print(f"  {cmd.method}: {cmd.args}")

## 5. Maxwell 3D 빌드 — 축대칭 단면을 360도 회전

DoSA 2D 데이터는 축대칭(X=반경, Y=축)이므로,  
3D 모드에서는 단면 sheet를 자동으로 Y축 기준 360도 회전시켜 솔리드를 생성합니다.

In [ ]:
# Solenoid를 3D 모델로 빌드 (revolve)
out_sol_3d = ROOT / "output" / "tutorial_solenoid_3d"
builder_3d = MaxwellSessionBuilder(
    design=design_sol,
    profile=profile_ws01,
    out_dir=out_sol_3d,
    mode="3d",
)
# live=True: 실제 AEDT 세션에 3D 프로젝트 생성
result_3d = builder_3d.build(live=True)

print(f"3D Build result: ok={result_3d.ok}, commands={len(result_3d.commands)}, errors={len(result_3d.errors)}")
if result_3d.errors:
    print("\nErrors:")
    for e in result_3d.errors:
        print(f"  - {e}")

In [ ]:
# 3D 빌드 커맨드 로그 확인 (sweep_around_axis 포함)
cmd_log_3d = json.loads((out_sol_3d / "maxwell_commands.json").read_text(encoding="utf-8"))

print(f"Total 3D commands: {len(cmd_log_3d['commands'])}")
print(f"Profile: {cmd_log_3d['profile']['name']} ({cmd_log_3d['profile']['solution_type']})")
print("\n3D command sequence:")
for i, cmd in enumerate(cmd_log_3d['commands'], 1):
    args_summary = ', '.join(f"{k}={v}" for k, v in cmd['args'].items() if k != 'points')
    print(f"  {i:2d}. {cmd['method']:32s} | {args_summary}")

In [ ]:
# VCM도 3D로 빌드 (자석 포함 모델)
out_vcm_3d = ROOT / "output" / "tutorial_vcm_3d"
builder_vcm_3d = MaxwellSessionBuilder(
    design=design_vcm,
    profile=get_profile("le01_2020r1"),
    out_dir=out_vcm_3d,
    mode="3d",
)
result_vcm_3d = builder_vcm_3d.build(live=True)

print(f"VCM 3D build: ok={result_vcm_3d.ok}, commands={len(result_vcm_3d.commands)}, errors={len(result_vcm_3d.errors)}")
if result_vcm_3d.errors:
    print("\nErrors:")
    for e in result_vcm_3d.errors:
        print(f"  - {e}")

## 6. Unified Plan 상태 확인

In [ ]:
from dosa_maxwell import get_unified_plan_summary

plan = get_unified_plan_summary()
print(f"Version: {plan['version']}")
print(f"Scope: {plan['scope']}")
print(f"\nSources:")
for k, v in plan['sources'].items():
    print(f"  {k}: {v}")
print(f"\nMilestones:")
for m in plan['milestones']:
    icon = '\u2705' if m['status'] == 'completed' else '\U0001f504' if m['status'] == 'in-progress' else '\u2b1c'
    print(f"  {icon} {m['id']}: {m['title']} [{m['status']}]")

## 다음 단계

- `live=True`로 실행하면 실제 AEDT 세션에 프로젝트가 생성됩니다.
- `--profile ws01_2020r1` 또는 `le01_2020r1`로 PDF 기반 프리셋을 CLI에서도 선택 가능합니다.
- `config/unified_plan.json`을 수정하면 새 프로파일을 추가하거나 마일스톤을 갱신할 수 있습니다.